In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import re
import json
import time
import pickle
import requests
from pathlib import Path

### Load Chunks

In [4]:
with open("../data/processed/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

In [5]:
chunks = bm25_index["chunks"]

### Generate a test question

In [6]:
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL = "http://localhost:11434/api/generate"

In [7]:
def generate_questions_for_chunk(
    chunk: dict, n: int = 2, max_retries: int = 3
) -> list[str]:
    prompt = f"""Aşağıdaki Türk hukuku metninden {n} adet soru üret.

    Kurallar:
    - Sorular SADECE bu metne bakılarak cevaplanabilir olmalı
    - Sorular Türkçe olmalı
    - Her soru ayrı satırda olmalı
    - Numara, tire, madde işareti KULLANMA
    - Soru dışında hiçbir şey yazma
    
    Metin (Karayolları Trafik Kanunu - Madde {chunk['metadata']['madde_no']}):
    {chunk['text'][:800]}
    
    Sorular:"""

    for attempt in range(max_retries):
        try:
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": OLLAMA_MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.3,
                        "num_predict": 200,
                    },
                },
                timeout=60,
            )
            raw = response.json()["response"].strip()

            # Temizle: boş satır, numara, tire kaldır
            questions = []
            for line in raw.split("\n"):
                line = line.strip()
                if not line:
                    continue
                # "1. ", "- ", "• " gibi prefix'leri kaldır
                import re

                line = re.sub(r"^[\d\.\-\•\*]+\s*", "", line)
                if line.endswith("?"):
                    questions.append(line)

            return questions[:n]  # en fazla n soru döndür

        except requests.exceptions.ConnectionError:
            print("Ollama çalışmıyor — 'ollama serve' komutunu çalıştır")
            return []
        except Exception as e:
            print(f"Hata (deneme {attempt+1}): {e}")
            time.sleep(2)

    return []

In [8]:
test_chunk = chunks[0]
test_chunk

{'chunk_id': 'trafik_kanunu_1',
 'text': 'KARAYOLLARI TRAFİK KANUNU123\nKanun Numarası : 2918\nKabul Tarihi : 13/10/1983\nYayımlandığı Resmî Gazete : Tarih: 18/10/1983 Sayı: 18195\nYayımlandığı Düstur : Tertip: 5 Cilt: 22 Sayfa: 687\nBİRİNCİ KISIM\nGenel Esaslar\nBİRİNCİ BÖLÜM\nAmaç ve Kapsam\nAmaç:',
 'metadata': {'madde_no': 1, 'char_count': 249}}

In [9]:
questions = generate_questions_for_chunk(test_chunk, n=2)
print("Test chunk soruları:")
for q in questions:
    print(f"  - {q}")

Test chunk soruları:
  - Karayolları Trafik Kanunu hangi tarihte kabul edilmiştir?
  - Karayolları Trafik Kanununda hangi kanun numarası ile yüklüdür?


### Build the eval dataset

In [10]:
def build_eval_dataset(
    chunks: list[dict],
    sample_size: int = 50,
    questions_per_chunk: int = 2,
    output_path: str = "evaluation/questions.json",
) -> list[dict]:
    import random

    random.seed(42)

    sampled = random.sample(chunks, min(sample_size, len(chunks)))
    dataset = []

    for i, chunk in enumerate(sampled):
        print(f"[{i+1}/{len(sampled)}] {chunk['chunk_id']}", end=" ... ")

        questions = generate_questions_for_chunk(chunk, n=questions_per_chunk)

        for q in questions:
            dataset.append(
                {
                    "question": q,
                    "relevant_chunk_id": chunk["chunk_id"],
                    "kanun": "Karayolları Trafik Kanunu",
                    "madde_no": chunk["metadata"]["madde_no"],
                }
            )

        print(f"{len(questions)} soru üretildi")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    print(f"\nToplam {len(dataset)} soru → {output_path}")
    return dataset

In [11]:
dataset = build_eval_dataset(
    chunks, sample_size=50, questions_per_chunk=2, output_path="evaluation/questions.json"
)

[1/50] trafik_kanunu_29 ... 2 soru üretildi
[2/50] trafik_kanunu_7 ... 2 soru üretildi
[3/50] trafik_kanunu_71 ... 2 soru üretildi
[4/50] trafik_kanunu_63 ... 2 soru üretildi
[5/50] trafik_kanunu_58 ... 2 soru üretildi
[6/50] trafik_kanunu_36 ... 2 soru üretildi
[7/50] trafik_kanunu_27 ... 2 soru üretildi
[8/50] trafik_kanunu_23 ... 1 soru üretildi
[9/50] trafik_kanunu_109 ... 0 soru üretildi
[10/50] trafik_kanunu_9 ... 2 soru üretildi
[11/50] trafik_kanunu_8 ... 2 soru üretildi
[12/50] trafik_kanunu_24 ... 2 soru üretildi
[13/50] trafik_kanunu_28 ... 2 soru üretildi
[14/50] trafik_kanunu_30 ... 2 soru üretildi
[15/50] trafik_kanunu_65 ... 2 soru üretildi
[16/50] trafik_kanunu_78 ... 2 soru üretildi
[17/50] trafik_kanunu_4 ... 2 soru üretildi
[18/50] trafik_kanunu_72 ... 2 soru üretildi
[19/50] trafik_kanunu_26 ... 2 soru üretildi
[20/50] trafik_kanunu_92 ... 2 soru üretildi
[21/50] trafik_kanunu_84 ... 2 soru üretildi
[22/50] trafik_kanunu_90 ... 2 soru üretildi
[23/50] trafik_kanunu_